# PDF Input Quickstart for Qwen3.5-27B

この Notebook は、英語論文 PDF の **全ページ** を対象にして、**text として入れる方法** と **画像化して vision input として入れる方法** をまとめたものです。


## 1. 前提
- Docker で SGLang API サーバが起動している
- `OPENAI_BASE_URL` を必要に応じて設定する
- PDF サンプルは `papers/` に配置済み


In [1]:
import os
from pathlib import Path
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:30000/v1')
PAPERS = Path('../papers')
print('Using base_url =', BASE_URL)
print('Papers:', sorted(p.name for p in PAPERS.glob('*.pdf')))
client = OpenAI(api_key='EMPTY', base_url=BASE_URL)
client


Using base_url = http://127.0.0.1:30009/v1
Papers: ['attention_is_all_you_need.pdf', 'deep_residual_learning.pdf']


## 2. PDF の全ページから text を抽出して入れる


In [2]:
from pypdf import PdfReader

pdf_path = PAPERS / 'attention_is_all_you_need.pdf'
reader = PdfReader(str(pdf_path))
text = '\n'.join(page.extract_text() or '' for page in reader.pages)
print('page_count =', len(reader.pages))
print('text_chars =', len(text))
print(text[:3000])


page_count = 15
text_chars = 39629
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recur

In [3]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'system', 'content': 'You are reading a full English deep learning paper extracted from PDF text.'},
        {'role': 'user', 'content': 'Summarize the paper in Japanese with sections for problem, method, and key ideas.\n\n' + text[:60000]},
    ],
    max_tokens=512,
)
resp


ChatCompletion(id='62d5a84deafa47f2ad923279e6d97718', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='Here\'s a thinking process that leads to the suggested summary:\n\n1.  **Analyze the Request:**\n    *   **Task:** Summarize the provided paper ("Attention Is All You Need") in Japanese.\n    *   **Structure:** Must include sections for "Problem" (課題), "Method" (手法), and "Key Ideas" (主要なアイデア).\n    *   **Input:** Full text of the paper (extracted from PDF).\n    *   **Constraint:** Proper attribution is granted for tables/figures (not directly relevant to the summary text itself, but good to note).\n    *   **Language:** Japanese.\n\n2.  **Analyze the Paper Content:**\n    *   **Title:** Attention Is All You Need.\n    *   **Authors:** Vaswani et al. (Google Brain/Research).\n    *   **Abstract:** Proposes "Tra

## 3. PDF の全ページを画像化して入れる


In [4]:
import fitz  # pymupdf

pdf_path = PAPERS / 'deep_residual_learning.pdf'
doc = fitz.open(pdf_path)
out_dir = PAPERS / 'rendered' / 'deep_residual_learning_all_pages'
out_dir.mkdir(parents=True, exist_ok=True)
image_paths = []
for i in range(len(doc)):
    page = doc.load_page(i)
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
    img_path = out_dir / f'deep_residual_learning_page_{i+1:02d}.png'
    pix.save(img_path)
    image_paths.append(img_path)
print('rendered_pages =', len(image_paths))
print(image_paths[:3], '...')


rendered_pages = 12
[PosixPath('../papers/rendered/deep_residual_learning_all_pages/deep_residual_learning_page_01.png'), PosixPath('../papers/rendered/deep_residual_learning_all_pages/deep_residual_learning_page_02.png'), PosixPath('../papers/rendered/deep_residual_learning_all_pages/deep_residual_learning_page_03.png')] ...


In [5]:
import base64
import mimetypes

content = [{'type': 'text', 'text': 'These are rendered pages from an English deep learning paper PDF. Summarize the paper in Japanese and mention the overall topic and architecture.'}]
for img_path in image_paths:
    mime = mimetypes.guess_type(img_path.name)[0] or 'image/png'
    image_url = 'data:' + mime + ';base64,' + base64.b64encode(img_path.read_bytes()).decode('utf-8')
    content.append({'type': 'image_url', 'image_url': {'url': image_url}})

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[{'role': 'user', 'content': content}],
    max_tokens=512,
)
resp


ChatCompletion(id='84421e78752a416cb9b7a2f3a65c657c', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='The user wants a summary of the provided paper "Deep Residual Learning for Image Recognition" in Japanese.\nI need to extract the following key information:\n1.  **Overall Topic:** What is the main problem being solved? (Training very deep neural networks, specifically the degradation problem).\n2.  **Architecture:** What is the proposed solution? (Residual Learning, Residual Networks/ResNets, shortcut connections).\n3.  **Key Results:** What were the outcomes? (Won ILSVRC 2015, significant accuracy improvements on ImageNet, CIFAR-10, COCO).\n\n**Drafting the summary (mental or scratchpad):**\n*   **Title:** Deep Residual Learning for Image Recognition (Microsoft Research).\n*   **Problem:** Deeper networks are

## 4. 補足
- 全ページ text 抽出は本文全体の要約向き
- 全ページ画像化は図表・レイアウト・ページ全体の理解向き
- 全ページ画像入力は重いので、必要に応じて一部ページだけに減らしてもよい
